# Agentic RAG với LangGraph — Pháp Lý Việt Nam

Triển khai **Agentic RAG** cho tra cứu văn bản pháp luật Việt Nam sử dụng LangGraph để xây dựng đồ thị agent có trạng thái. Pipeline gồm hai công cụ (vector DB nội bộ + web dự phòng) và vòng lặp suy luận tự động:
1. Agent nhận câu hỏi và quyết định gọi công cụ nào
2. `ToolNode` thực thi công cụ và trả kết quả về cho agent
3. Agent tiếp tục suy luận cho đến khi tổng hợp được câu trả lời cuối

In [6]:
%pip install -Uq langgraph

Note: you may need to restart the kernel to use updated packages.


## Bước 1 — Cài Đặt & Cấu Hình

Cài `langgraph` và thiết lập các biến môi trường kết nối đến LLM endpoint.

In [7]:
import os
import math
import json
from typing import Annotated, List
from typing_extensions import TypedDict

from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings

from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition


# ============================================================
# 1. Config
# ============================================================

LLM_URL = os.environ.get("LLM_URL", "https://api.openai.com/v1")
LLM_API_KEY = os.environ.get("LLM_API_KEY") or os.environ.get("OPENAI_API_KEY")
LLM_MODEL = os.environ.get("LLM_MODEL", "gpt-4o-mini")


## Bước 2 — Vector Store In-Memory

`TinyVectorDB` mã hóa tài liệu pháp lý bằng `HuggingFaceEmbeddings` (BAAI/bge-m3) và truy xuất qua độ tương đồng cosine — lớp tìm kiếm ngữ nghĩa nhẹ, không cần thư viện vector DB bên ngoài.

In [8]:
# ============================================================
# 2. Tiny in-memory vector DB
# ============================================================

def cosine_similarity(v1, v2):
    dot_product = sum(x * y for x, y in zip(v1, v2))
    magnitude1 = math.sqrt(sum(x * x for x in v1))
    magnitude2 = math.sqrt(sum(x * x for x in v2))

    if magnitude1 == 0 or magnitude2 == 0:
        return 0.0

    return dot_product / (magnitude1 * magnitude2)


class TinyVectorDB:
    def __init__(self):
        self.knowledge_base = []
        self.encoder = HuggingFaceEmbeddings(
            model_name="BAAI/bge-m3",
            encode_kwargs={"normalize_embeddings": True},
        )

    def add_documents(self, docs: List[str]):
        embeddings = self.encoder.embed_documents(docs)

        for doc, emb in zip(docs, embeddings):
            self.knowledge_base.append(
                {
                    "text": doc,
                    "embedding": emb,
                }
            )

    def query(self, query_text: str, top_k: int = 2):
        query_emb = self.encoder.embed_query(query_text)

        results = []
        for item in self.knowledge_base:
            score = cosine_similarity(query_emb, item["embedding"])
            results.append(
                {
                    "text": item["text"],
                    "score": score,
                }
            )

        results.sort(key=lambda x: x["score"], reverse=True)
        return results[:top_k]


db = TinyVectorDB()

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

## Bước 3 — Công Cụ Agent

Hai hàm `@tool` cung cấp khả năng truy xuất cho LLM:
- **`internal_vector_search`** — truy vấn vector DB nội bộ chứa văn bản pháp luật Việt Nam; luôn được gọi trước
- **`web_search`** — tìm kiếm web dự phòng khi kết quả nội bộ không đủ

In [ ]:
# ============================================================
# 3. Công cụ (Tools)
# ============================================================

@tool
def internal_vector_search(query: str) -> str:
    """
    Tìm kiếm trong cơ sở tri thức nội bộ về văn bản pháp luật Việt Nam.
    Luôn dùng công cụ này trước khi tìm kiếm web.
    Phù hợp cho các câu hỏi về luật, nghị định, thông tư, điều khoản.
    """
    print(f"\n[Công cụ] internal_vector_search: {query}")

    results = db.query(query, top_k=2)

    return json.dumps(
        {
            "source": "internal_vector_db",
            "query": query,
            "retrieved_docs": results,
        },
        ensure_ascii=False,
    )


@tool
def web_search(query: str) -> str:
    """
    Công cụ tìm kiếm dự phòng.
    Chỉ dùng khi vector database nội bộ không có đủ thông tin.
    """
    print(f"\n[Công cụ] web_search: {query}")

    if "hiệu lực" in query.lower() or "còn hiệu lực" in query.lower():
        return json.dumps(
            {
                "source": "simulated_web",
                "query": query,
                "result": "Theo Cổng thông tin pháp điển quốc gia, văn bản này hiện còn hiệu lực thi hành.",
            },
            ensure_ascii=False,
        )

    return json.dumps(
        {
            "source": "simulated_web",
            "query": query,
            "result": "Không tìm thấy thông tin pháp lý bên ngoài phù hợp.",
        },
        ensure_ascii=False,
    )


tools = [internal_vector_search, web_search]

## Bước 4 — Định Nghĩa Trạng Thái LangGraph

`AgentState` là schema trạng thái của đồ thị. Trường `messages` sử dụng reducer `add_messages` để tự động nối tin nhắn mới vào lịch sử thay vì ghi đè.

In [10]:
# ============================================================
# 4. LangGraph state
# ============================================================

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]


## Bước 5 — LLM & System Prompt

`ChatOpenAI` được khởi tạo và `.bind_tools()` đăng ký các công cụ vào không gian hành động của model. `SYSTEM_PROMPT` định hướng agent ưu tiên tra cứu nội bộ, trích dẫn điều khoản cụ thể và trả lời bằng tiếng Việt.

In [ ]:
# ============================================================
# 5. LLM
# ============================================================

llm = ChatOpenAI(
    base_url=LLM_URL,
    api_key=LLM_API_KEY,
    model=LLM_MODEL,
    temperature=0,
).bind_tools(tools)


SYSTEM_PROMPT = SystemMessage(
    content="""
Bạn là trợ lý Agentic RAG chuyên về pháp luật Việt Nam.

Bạn có hai công cụ:
1. internal_vector_search
2. web_search

Quy tắc:
- Luôn tìm kiếm vector database nội bộ trước.
- Kiểm tra tài liệu được trả về và điểm số liên quan.
- Nếu kết quả nội bộ không liên quan, không đầy đủ hoặc yếu, hãy dùng web_search.
- Không trả lời chỉ từ kiến thức có sẵn của model.
- Câu trả lời cuối phải nêu rõ thông tin đến từ tìm kiếm nội bộ hay web.
- Nếu không có nguồn nào đủ bằng chứng, hãy nói rõ ngữ cảnh hiện có không đủ để trả lời.
- Trả lời bằng tiếng Việt, ngắn gọn và trích dẫn điều khoản cụ thể.
"""
)

## Bước 6 — Các Node trong Đồ Thị

Hai node tạo thành vòng lặp suy luận:
- **`agent_node`** — gọi LLM với toàn bộ lịch sử tin nhắn; quyết định gọi công cụ hoặc trả câu trả lời cuối
- **`tool_node`** — `ToolNode` của LangGraph tự động thực thi tool call và trả kết quả

In [12]:
# ============================================================
# 6. Graph nodes
# ============================================================

def agent_node(state: AgentState):
    """
    LLM reasoning node.
    It decides whether to call a tool or produce the final answer.
    """
    response = llm.invoke([SYSTEM_PROMPT] + state["messages"])
    return {"messages": [response]}


tool_node = ToolNode(tools)


## Bước 7 — Xây Dựng Đồ Thị LangGraph

Kết nối các node thành đồ thị có điều kiện:
- `START → agent` — bắt đầu từ node suy luận
- `agent → tools` (có điều kiện) — nếu agent phát sinh tool call thì chuyển sang node công cụ
- `agent → END` — nếu không có tool call thì kết thúc và trả câu trả lời
- `tools → agent` — sau khi công cụ chạy xong, quay lại agent để tiếp tục suy luận

In [13]:
# ============================================================
# 7. Build LangGraph
# ============================================================

graph_builder = StateGraph(AgentState)

graph_builder.add_node("agent", agent_node)
graph_builder.add_node("tools", tool_node)

graph_builder.add_edge(START, "agent")

# If the agent produced tool calls, go to tools.
# Otherwise, stop and return final answer.
graph_builder.add_conditional_edges("agent", tools_condition)

# After tools run, return to the agent for another reasoning step.
graph_builder.add_edge("tools", "agent")

agentic_rag_graph = graph_builder.compile()

## Bước 8 — Hàm Thực Thi Agent

`run_agentic_rag` bọc lệnh `invoke` của đồ thị đã biên dịch, truyền câu hỏi người dùng dưới dạng `HumanMessage` và trả về nội dung tin nhắn cuối cùng.

In [14]:
def run_agentic_rag(question: str):
    result = agentic_rag_graph.invoke(
        {
            "messages": [
                HumanMessage(content=question)
            ]
        },
        config={
            "recursion_limit": 10
        },
    )

    final_message = result["messages"][-1]
    return final_message.content

## Bước 9 — Cơ Sở Tri Thức & Demo

Nạp văn bản pháp lý Việt Nam vào vector DB và chạy hai kịch bản:
1. **Truy xuất nội bộ** — câu hỏi về giờ làm thêm được tìm thấy trực tiếp trong DB
2. **Dự phòng web** — câu hỏi về hiệu lực văn bản không có trong DB, agent tự động chuyển sang `web_search`

In [ ]:
db.add_documents(
    [
        "Theo Điều 105 Bộ luật Lao động 2019, thời giờ làm việc bình thường không quá 8 giờ trong một ngày và không quá 48 giờ trong một tuần.",
        "Điều 107 Bộ luật Lao động 2019 quy định người lao động làm thêm giờ không được vượt quá 50% số giờ làm việc bình thường trong ngày; tổng số giờ làm việc và làm thêm không quá 12 giờ trong một ngày.",
        "Điều 111 Luật Doanh nghiệp 2020 quy định công ty cổ phần là doanh nghiệp có vốn điều lệ được chia thành cổ phần; số lượng cổ đông tối thiểu là 03 và không hạn chế số lượng tối đa.",
        "Khoản 1 Điều 166 Luật Đất đai 2013 quy định người sử dụng đất có quyền được cấp Giấy chứng nhận quyền sử dụng đất, quyền sở hữu nhà ở và tài sản khác gắn liền với đất.",
    ]
)

print("\n--- Demo 1: Truy xuất nội bộ ---")
answer = run_agentic_rag("Người lao động được làm thêm tối đa bao nhiêu giờ mỗi ngày?")
print(answer)

print("\n" + "=" * 80)

print("\n--- Demo 2: Dự phòng web ---")
answer = run_agentic_rag("Luật Đất đai 2013 hiện còn hiệu lực không?")
print(answer)